# Solution of linear systems and interpolations

Using numerical-computational techniques to solve linear systems and interpolations 💻

## All imports:

Import the dependencies required for the project

In [70]:
import numpy as np
import sys
from typing import Tuple, Callable

## Solving Linear Systems:

Solving linear systems using direct and iterative numerical methods

### Overall parameters

Pre-defined parameters for analysis

In [71]:
def create_dominant_diagonal_matrix(n: int) -> np.ndarray:
    matrix = np.random.randint(low=0, high=10, size=(n, n))
    for i in range(n):
        matrix[i,i] = np.sum(abs(matrix[i, :])) + np.sum(abs(matrix[:, i])) + 1
    return matrix

A : np.ndarray = np.array([
        [10, 2, 1],
        [1, 5, 1],
        [2, 3, 10]
    ])

F : np.ndarray= np.array([
        [32,  5,  7,  0],
        [ 3, 30,  6,  1],
        [ 7,  5, 54,  8],
        [ 1,  9,  8, 30]
    ])

C : np.ndarray= np.array([
        [45,  2,  5,  6,  0],
        [ 9, 37,  1,  7,  3],
        [ 6,  1, 49,  8,  5],
        [ 8,  9,  7, 55,  1],
        [ 0,  0,  3,  2, 31]
    ])

B3 : np.ndarray = np.array([7, -8, 6])
B4 : np.ndarray = np.array([1,2,3,4])
B5 : np.ndarray = np.array([1,2,3,4,5])

epsilon = 10**(-16)

### Auxiliary Functions

Auxiliary functions are utilities that perform specific and repetitive tasks, simplifying the main code. Below are the functions used in this project:

In [72]:
def dominant_line(matrix: np.ndarray) -> bool:
    for i in range(matrix.shape[0]):
        diagonal = np.abs(matrix[i, i])
        sum_line = np.sum(np.abs(matrix[i, :])) - diagonal
        if diagonal < sum_line:
            return False
    return True

def dominant_column(matrix: np.ndarray) -> bool:
    for i in range(matrix.shape[1]):
        diagonal = np.abs(matrix[i, i])
        sum_column = np.sum(np.abs(matrix[:, i])) - diagonal
        if diagonal < sum_column:
            return False
    return True

def dominant_diagonal(matrix: np.ndarray) -> bool:
    return dominant_line(matrix) or dominant_column(matrix)

def absolute_distance(new: np.ndarray, old: np.ndarray) -> np.float64:
    return np.max(abs(new-old))

def relative_distance(new: np.ndarray, old: np.ndarray, epsilon: float = 0) -> np.float64:
    ad = absolute_distance(new,old)
    new_max = np.max(abs(new))
    
    if new_max <= epsilon:
        new_max+=sys.float_info.epsilon
    
    return ad/new_max

def residue(A: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.float64:
    return np.max(abs(b - (A @ x)))


### Gauss-Elimination

In [73]:
def gauss_scaling(A: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    """Performs Gaussian elimination with partial pivoting and scaling on a linear system.

    Transforms the coefficient matrix into upper triangular form using row operations,
    while maintaining numerical stability through partial pivoting. Also computes the
    determinant of the matrix as a side product.

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        Tuple containing:
        - A_new: Upper triangular matrix after elimination
        - b_new: Modified right-hand side vector after elimination
        - det: Determinant of the original matrix (product of diagonal elements with sign changes)

    Raises:
        ValueError: If the matrix is singular (no unique solution exists).

    Note:
        Modifies the input matrix and vector during the elimination process.
        The determinant is computed as a side product of the elimination steps.
    """
    n = len(b)
    det = 1.0

    for k in range(n):
        pivot_selection = np.argmax(abs(A[k:, k])) + k

        if A[pivot_selection, k] < sys.float_info.epsilon:
            raise ValueError("Matrix is singular (no unique solution).")

        if pivot_selection != k:
            A[[k, pivot_selection]] = A[[pivot_selection, k]]
            b[[k, pivot_selection]] = b[[pivot_selection, k]]
            det *= -1

        for i in range(k + 1, n):
            factor = A[i, k] / A[k, k]
            A[i, k:] -= factor * A[k, k:]
            b[i] -= factor * b[k]

    det *= np.prod(np.diagonal(A))
    return A, b, det

def retrosubstitution(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solves an upper triangular linear system using backward substitution.

    Args:
        A: Upper triangular coefficient matrix (n x n numpy array).
        b: Right-hand side vector (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If any diagonal element is zero (matrix is singular).
    """
    n = len(b)
    x = np.zeros(n)
    
    for i in reversed(range(n)):
        soma = np.dot(A[i, i+1:], x[i+1:])
        x[i] = (b[i] - soma) / A[i, i]
    
    return x

def gauss_elimination_method(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Solves a system of linear equations using Gaussian elimination with partial pivoting.

    Performs the complete solution process:
    1. Transforms the matrix to upper triangular form with partial pivoting
    2. Checks for singularity (zero determinant)
    3. Solves the triangular system using backward substitution

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix is singular (determinant is zero) or if any diagonal
                   element becomes zero during elimination.
    """

    A = A.astype(float)
    b = b.astype(float)
    
    A_new, b_new, det = gauss_scaling(A,b)

    if det == 0:
        raise ValueError("No single solution")
    
    return retrosubstitution(A_new, b_new)


### LU Factorization Method

In [74]:
def lu_decomposition(A):
    """Performs LU decomposition with partial pivoting on a square matrix.

    Decomposes a matrix A into PA = LU, where:
    - P is a permutation matrix
    - L is a lower triangular matrix with unit diagonal
    - U is an upper triangular matrix

    Args:
        A: Square coefficient matrix to decompose (n x n numpy array).

    Returns:
        Tuple containing three numpy arrays:
        - P: Permutation matrix representing row exchanges
        - L: Lower triangular matrix with ones on diagonal
        - U: Upper triangular matrix

    Raises:
        ValueError: If the matrix is singular (no unique decomposition exists).

    Note:
        Uses partial pivoting for numerical stability.
        The decomposition satisfies PA = LU.
    """
    A = A.astype(float)
    n = A.shape[0]
    
    L = np.eye(n)
    U = A.copy()
    P = np.eye(n)

    for k in range(n):
       
        pivot_row = np.argmax(np.abs(U[k:, k])) + k
        
        if U[pivot_row, k] == 0:
            raise ValueError("Matrix is singular (no unique solution).")
        
       
        if pivot_row != k:
            U[[k, pivot_row], :] = U[[pivot_row, k], :]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]
    
        for i in range(k + 1, n):
            fator = U[i, k] / U[k, k]
            L[i, k] = fator
            U[i, :] -= fator * U[k, :]

    return P, L, U

def lu_method(A, b):
    """Solves a linear system using LU decomposition with partial pivoting.

    Solves Ax = b by:
    1. Decomposing A into PA = LU
    2. Solving Ly = Pb (forward substitution)
    3. Solving Ux = y (backward substitution)

    Args:
        A: Square coefficient matrix of the linear system (n x n numpy array).
        b: Right-hand side vector of the linear system (n-dimensional numpy array).

    Returns:
        np.ndarray: Solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix is singular (no unique solution exists).

    Note:
        More efficient than Gaussian elimination when solving multiple systems
        with the same coefficient matrix but different right-hand sides.
    """
    P, L, U = lu_decomposition(A)
    Pb = P @ b
    
    n = A.shape[0]
    
    
    y = np.zeros_like(b, dtype=float)
    for i in range(n):
        y[i] = Pb[i] - L[i, :i] @ y[:i]
        
    
    x = np.zeros_like(b, dtype=float)
    for i in reversed(range(n)):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
        
    return x


### Gauss-Jacobi

In [75]:
def gauss_jacobi_method(A: np.ndarray, b: np.ndarray, max_iterations: int = 100, epsilon: float = sys.float_info.epsilon) -> np.ndarray:
    """Solves a system of linear equations using the Gauss-Jacobi iterative method.

    The Gauss-Jacobi method is an iterative algorithm for solving systems of linear equations
    where the matrix is strictly diagonally dominant or symmetric and positive definite.
    At each iteration, it uses the solution from the previous iteration to compute new values.

    Args:
        A: A square numpy array representing the coefficient matrix of the linear system.
            Must be strictly diagonally dominant for convergence guarantee.
            Shape: (n, n) where n is the number of equations.
        b: A numpy array representing the right-hand side vector of the linear system.
            Shape: (n,) where n matches the matrix dimension.
        max_iterations: Maximum number of iterations to perform before giving up.
            Default: 100.
        epsilon: The tolerance for determining convergence. The algorithm stops when either
            the absolute or relative distance between iterations is less than epsilon.
            Default: system's float epsilon.

    Returns:
        A numpy array representing the solution vector x that satisfies Ax = b.
        Shape: (n,) matching the input dimensions.

    Raises:
        ValueError: If any of the following occurs:
            - Matrix doesn't have a dominant diagonal (no convergence guarantee)
            - Any diagonal element is zero (would cause division by zero)
            - Maximum number of iterations is exceeded without convergence

    Note:
        Requires two helper functions:
        1. dominant_diagonal(): Checks if matrix is diagonally dominant
        2. absolute_distance() and relative_distance(): Calculate convergence criteria
    """
    if not dominant_diagonal(A):
        raise ValueError("Does not guarantee convergence")
    
    if np.any(np.diag(A) == 0):
        raise ValueError("Zero on diagonal - cannot divide by zero")
    
    n = A.shape[0]
    x = np.array([b[i]/A[i, i] for i in range(n)])
    
    for _ in range(max_iterations):
        x_old = x.copy()
        
        for i in range(n):
            sum = np.sum(A[i, :i] @ x_old[:i]) + (A[i, i+1:] @ x_old[i+1:])
            x[i] = (b[i] - sum) / A[i, i]
        
        if absolute_distance(x,x_old) < epsilon or relative_distance(x,x_old,epsilon) < epsilon:
            return x
    
    raise ValueError("Exceeded the maximum number of iterations")

### Gauss-Seidel

In [76]:
def gauss_seidel_method(A: np.ndarray, b: np.ndarray, max_iterations: int = 100, epsilon: float = sys.float_info.epsilon) -> np.ndarray:
    """Solves a system of linear equations using the Gauss-Seidel iterative method.

    The Gauss-Seidel method is an iterative technique for solving a square system of n linear
    equations with unknown x. The method will converge if the matrix is either strictly diagonally
    dominant or symmetric and positive definite.

    Args:
        A: A square numpy array representing the coefficient matrix of the linear system.
            Must be strictly diagonally dominant for convergence guarantee.
        b: A numpy array representing the right-hand side vector of the linear system.
        max_iterations: Maximum number of iterations to perform before giving up.
            Defaults to 100.
        epsilon: The tolerance for determining convergence. The algorithm stops when either
            the absolute or relative distance between iterations is less than epsilon.
            Defaults to system's float epsilon.

    Returns:
        A numpy array representing the solution vector x that satisfies Ax = b.

    Raises:
        ValueError: If the matrix doesn't have a dominant diagonal (no convergence guarantee),
            if any diagonal element is zero (would cause division by zero),
            or if the maximum number of iterations is exceeded without convergence.

    Note:
        The function uses two helper functions:
        - dominant_diagonal(): Checks if matrix is diagonally dominant
        - absolute_distance() and relative_distance(): Calculate convergence criteria
    """
    if not dominant_diagonal(A):
        raise ValueError("Does not guarantee convergence")
    
    if np.any(np.diag(A) == 0):
        raise ValueError("Zero on diagonal - cannot divide by zero")
    
    n = A.shape[0]
    x = np.array([b[i]/A[i, i] for i in range(n)])
    
    for _ in range(max_iterations):
        x_old = x.copy()
        
        for i in range(n):
            sum = np.sum(A[i, :i] @ x[:i]) + (A[i, i+1:] @ x[i+1:])
            x[i] = (b[i] - sum) / A[i, i]
        
        if absolute_distance(x,x_old) < epsilon or relative_distance(x,x_old,epsilon) < epsilon:
            return x
    
    raise ValueError("Exceeded the maximum number of iterations")

### Test

In [77]:
# Matrix A
print(f"Matrix A:\n{A}\nb-vector:\n{B3}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(A, B3)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

print("LU Decomposition Method:")
x = lu_method(A, B3)
print(f"Solution: {x}")
print(f"Residue: {residue(A, B3, x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(A, B3, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(A, B3, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(A,B3,x)}\n")

Matrix A:
[[10  2  1]
 [ 1  5  1]
 [ 2  3 10]]
b-vector:
[ 7 -8  6]

Gauss-Elimination Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

LU Decomposition Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

Gauss-Jacobi Method:
Solution: [ 1. -2.  1.]
Residue: 0.0

Gauss-Seidel Method:
Solution: [ 1. -2.  1.]
Residue: 0.0



In [78]:
# Matrix F
print(f"Matrix F:\n{F}\nb-vector:\n{B4}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(F, B4)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("LU Decomposition Method:")
x = lu_method(F, B4)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(F, B4, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(F, B4, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(F,B4,x)}\n")

Matrix F:
[[32  5  7  0]
 [ 3 30  6  1]
 [ 7  5 54  8]
 [ 1  9  8 30]]
b-vector:
[1 2 3 4]

Gauss-Elimination Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 0.0

LU Decomposition Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 0.0

Gauss-Jacobi Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 8.881784197001252e-16

Gauss-Seidel Method:
Solution: [0.01554328 0.0550245  0.03249894 0.10764149]
Residue: 2.220446049250313e-16



In [79]:
# Matrix C
print(f"Matrix C:\n{C}\nb-vector:\n{B5}\n")

print("Gauss-Elimination Method:")
x = gauss_elimination_method(C, B5)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("LU Decomposition Method:")
x = lu_method(C, B5)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("Gauss-Jacobi Method:")
x = gauss_jacobi_method(C, B5, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

print("Gauss-Seidel Method:")
x = gauss_seidel_method(C, B5, epsilon=epsilon)
print(f"Solution: {x}")
print(f"Residue: {residue(C,B5,x)}\n")

Matrix C:
[[45  2  5  6  0]
 [ 9 37  1  7  3]
 [ 6  1 49  8  5]
 [ 8  9  7 55  1]
 [ 0  0  3  2 31]]
b-vector:
[1 2 3 4 5]

Gauss-Elimination Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16

LU Decomposition Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16

Gauss-Jacobi Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 1.7763568394002505e-15

Gauss-Seidel Method:
Solution: [0.00926115 0.02706719 0.03404403 0.05981566 0.15413666]
Residue: 8.881784197001252e-16



## Interpolations:

Techniques for estimating unknown values between points

### Overall parameters

In [80]:
x_points = np.array([-1, 0, 2])
y_points = np.array([4, 1, -1])

### Linear System Method

#### Vandemond Matrix Function

In [ ]:
def vandermonde_matrix(x: np.ndarray) -> np.ndarray:
    """Constructs a Vandermonde matrix from given x-coordinates.
    
    A Vandermonde matrix is a matrix with terms of a geometric progression in each row.
    For interpolation, it's used to find coefficients of the interpolating polynomial.
    The matrix has the form:
        [[1, x₁, x₁², ..., x₁ⁿ],
         [1, x₂, x₂², ..., x₂ⁿ],
         ...
         [1, xₘ, xₘ², ..., xₘⁿ]]

    Args:
        x: A 1-D numpy array of x-coordinates.
            Shape: (n,) where n is the number of points.
            These are the points at which we know the function values.

    Returns:
        np.ndarray: The Vandermonde matrix.
            Shape: (n, n) where n is len(x).
            Each row contains increasing powers of the corresponding x value.

    Note:
        - The degree of the polynomial is len(x) - 1
        - The j-th column contains x values raised to power (j-1)
        - This matrix is often ill-conditioned for high degrees
    """
    degree = len(x) - 1
    return np.array([[xi**j for j in range(degree + 1)] for xi in x])

#### Linear Method

In [ ]:
def linear_method(x_points: np.ndarray, y_points: np.ndarray) -> np.ndarray:
    """Solves a polynomial interpolation problem using linear system method.

    This method converts the interpolation problem into a linear system using
    the Vandermonde matrix and solves it using LU decomposition to find the
    coefficients of the interpolating polynomial.

    Args:
        x_points: A 1-D numpy array of x-coordinates of the data points.
            Shape: (n,) where n is the number of points.
        y_points: A 1-D numpy array of y-coordinates of the data points.
            Shape: (n,) must match x_points shape.

    Returns:
        np.ndarray: Coefficients of the interpolating polynomial in ascending
            order of degree (constant term first).

    Raises:
        ValueError: If x_points and y_points have different lengths or if
            the resulting Vandermonde matrix is singular.

    Note:
        Uses vandermonde_matrix() to construct the coefficient matrix and
        lu_method() to solve the resulting linear system.
    """
    if x_points.shape[0] != y_points.shape[0]:
            raise ValueError("x_points and y_points have different lengths")
    V_matrix = vandermonde_matrix(x_points)
    A_points = lu_method(V_matrix, y_points)
    return A_points

#### Tests

In [83]:
x_points = np.array([-1, 0, 2])
y_points = np.array([4, 1, -1])

A = linear_method(x_points, y_points)
print("Coefficients:", A)

Coefficients: [ 1.         -2.33333333  0.66666667]


### Lagrange Method

In [84]:
def lagrange_method(x_points: np.ndarray, y_points: np.ndarray) -> Callable[[float], float]:
    """Constructs and returns the Lagrange interpolation polynomial for given data points.

    The Lagrange interpolation polynomial is the unique polynomial of least degree that
    interpolates a given set of points (x_i, y_i) and passes through all of them.

    Args:
        x_points: A 1-D numpy array of x-coordinates of the data points (interpolation nodes).
            Must not contain duplicate values.
        y_points: A 1-D numpy array of y-coordinates of the data points corresponding to x_points.
            Must have the same length as x_points.

    Returns:
        A callable function that represents the Lagrange interpolation polynomial.
        The function takes a float value x and returns the interpolated value at x.

    Raises:
        ValueError: If x_points and y_points have different lengths
    """
    if x_points.shape[0] != y_points.shape[0]:
            raise ValueError("x_points and y_points have different lengths")
    def polynomial(x: float) -> float:
        """Evaluates the Lagrange interpolation polynomial at point x.

        Args:
            x: The point at which to evaluate the interpolation polynomial.

        Returns:
            The interpolated value at point x.

        Note:
            This is the actual implementation of the Lagrange interpolation formula:
            L(x) = Σ [y_i * ℓ_i(x)] where ℓ_i(x) is the i-th Lagrange basis polynomial.
        """
        
        n = x_points.shape[0]
        result = 0.0
        for i in range(n):
            prod = 1.0
            for j in range(n):
                if i == j:
                    continue
                prod *= (x - x_points[j]) / (x_points[i] - x_points[j])
            result += y_points[i] * prod
        return result
    return polynomial

#### Tests

In [85]:
polynomial = lagrange_method(x_points=x_points, y_points=y_points)
print(polynomial(-1))
print(polynomial(0))
print(polynomial(2))
print(polynomial(1))

4.0
1.0
-1.0
-0.6666666666666665


### Newton Method


In [86]:
def newton_div_diff(x,y):
    """
    Computes the coefficients of the Newton interpolating polynomial 
    using divided differences.

    Given data points (x, y), this function returns the coefficients 
    of the Newton form of the interpolating polynomial.

    Parameters
    ----------
    x : array-like
        Array of x-values (independent variable).
    y : array-like
        Array of y-values (dependent variable) corresponding to x.

    Returns
    -------
    coef : ndarray
        Array of divided difference coefficients for the Newton interpolating polynomial.
    """   
    n = len(x)
    coef = np.copy(y).astype(float)
    for j in range(1, n):
        for i in range(n-1, j-1, -1):
            coef[i] = (coef[i] - coef[i-1]) / (x[i] - x[i-j])
    return coef


def newton_poly(coef, a, x):

    """
    Evaluates the Newton interpolating polynomial at a given point.

    This function uses the divided difference coefficients to compute 
    the value of the Newton interpolating polynomial at a specified x.

    Parameters
    ----------
    coef : array-like
        Array of divided difference coefficients computed by newton_div_diff().
    a : array-like
        Array of x-values used to compute the divided differences.
    x : float
        The point at which the polynomial is evaluated.

    Returns
    -------
    poly_value : float
        The interpolated value of the polynomial at point x.
    """
    n = len(coef) - 1
    poly_value = coef[n]
    for i in range(n-1, -1, -1):
        poly_value = coef[i] + (x - a[i]) * poly_value
    return poly_value

#### Tests

In [87]:
x_points = np.array([-1, 0, 2])
y_points = np.array([4, 1, -1])

coef = newton_div_diff(x_points, y_points)

print("Coeficientes:", coef)


for xi, yi in zip(x_points, y_points):
    y_interp = newton_poly(coef, x_points, xi)
    print(f"P({xi}) = {y_interp} (esperado: {yi})")

Coeficientes: [ 4.         -3.          0.66666667]
P(-1) = 4.0 (esperado: 4)
P(0) = 1.0 (esperado: 1)
P(2) = -1.0 (esperado: -1)


## Solutions of the avaliation

### Solve all the linear systems in the list of exercises using the methods requested in each question.

#### 15. Solve the following systems of linear equations using the Gauss elimination and LU factorization methods, using pivoting when deemed necessary.

### (a)
$$
\begin{bmatrix}
-9 & 5 & 6 \\
2 & 3 & 1 \\
-1 & 1 & -3
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix} =
\begin{bmatrix}
11 \\
4 \\
-2
\end{bmatrix}
$$

### (b)
$$
\begin{bmatrix}
2 & -1 & 1 \\
3 & 3 & 9 \\
3 & 3 & 5
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix} =
\begin{bmatrix}
-1 \\
0 \\
4
\end{bmatrix}
$$

### (c)
$$
\begin{bmatrix}
0.252 & 0.36 & 0.12 \\
0.112 & 0.16 & 0.24 \\
0.147 & 0.21 & 0.25
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix} =
\begin{bmatrix}
7 \\
8 \\
9
\end{bmatrix}
$$

### (d)
$$
\begin{bmatrix}
3 & -2 & 5 & 1 \\
-6 & 4 & -8 & 1 \\
9 & -6 & 19 & 1 \\
6 & -4 & -6 & 15
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix} =
\begin{bmatrix}
7 \\
-9 \\
23 \\
11
\end{bmatrix}
$$

##### Solutions for Gauss Elimination and LU factorization

In [107]:
A15a = np.array([
    [-9, 5, 6],
    [2, 3, 1],
    [-1, 1, -3]
])
b15a = np.array([11, 4, -2])

# Question 15(b)
A15b = np.array([
    [2, -1, 1],
    [3, 3, 9],
    [3, 3, 5]
])
b15b = np.array([-1, 0, 4])

# Question 15(c)
A15c = np.array([
    [0.252, 0.36, 0.12],
    [0.112, 0.16, 0.24],
    [0.147, 0.21, 0.25]
])
b15c = np.array([7, 8, 9])

# Question 15(d)
A15d = np.array([
    [3, -2, 5, 1],
    [-6, 4, -8, 1],
    [9, -6, 19, 1],
    [6, -4, -6, 15]
])
b15d = np.array([7, -9, 23, 11])

# Systems
systems = [
    ("15(a)", A15a, b15a),
    ("15(b)", A15b, b15b),
    ("15(c)", A15c, b15c),
    ("15(d)", A15d, b15d)
]

# Systems
systems = [
    ("15(a)", A15a, b15a),
    ("15(b)", A15b, b15b),
    ("15(c)", A15c, b15c),
    ("15(d)", A15d, b15d)
]

for name, A, b in systems:
    print(f"\nSolving {name}")
    
    # Gaussian elimination
    try:
        x_gauss = gauss_elimination_method(A, b)
        print(f"Gaussian elimination solution: {x_gauss}")
        print(f"Residue: {residue(A,b,x_gauss)}\n")
    except ValueError as e:
        print(f"Gaussian elimination failed: {e}")
    
    # LU factorization
    try:
        x_lu = lu_method(A, b)
        print(f"LU factorization solution: {x_lu}")
        print(f"Residue: {residue(A,b,x_lu)}\n")
    except ValueError as e:
        print(f"LU factorization failed: {e}")


Solving 15(a)
Gaussian elimination failed: Matrix is singular (no unique solution).
LU factorization solution: [-0.  1.  1.]
Residue: 8.881784197001252e-16


Solving 15(b)
Gaussian elimination failed: Matrix is singular (no unique solution).
LU factorization solution: [ 1.  2. -1.]
Residue: 0.0


Solving 15(c)
Gaussian elimination failed: Matrix is singular (no unique solution).
LU factorization solution: [-1.04164889e+16  7.29154225e+15  2.61904762e+01]
Residue: 0.31635374149659956


Solving 15(d)
Gaussian elimination failed: Matrix is singular (no unique solution).
LU factorization failed: Matrix is singular (no unique solution).


#### 16. Solve the following systems of linear equations using the Gauss-Jacobi and Gauss-Seidel methods, with a stopping criterion of stopping criterion $ε = 10^{-3}$ or a limit of 5 iterations, and as initial solution the relation $\frac{bi}{aii}$.

### (a)
$$
\begin{bmatrix}
10 & -1 & 2 & 0 \\
-1 & 11 & -1 & 3 \\
2 & -1 & 10 & -1 \\
0 & 3 & -1 & 8
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix} =
\begin{bmatrix}
6 \\
25 \\
-11 \\
15
\end{bmatrix}
$$

### (b)
$$
\begin{bmatrix}
0 & 5 & -1 & 2 \\
0 & 8 & -1 & 1 \\
2 & 1 & -1 & -1 \\
0 & -1 & -2 & 1
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix} =
\begin{bmatrix}
10 \\
16 \\
2 \\
-2
\end{bmatrix}
$$

### (c)
$$
\begin{bmatrix}
1 & 0.5 & -0.1 & 0.1 \\
0.2 & 1 & -0.2 & -0.1 \\
-0.1 & -0.2 & 1 & 0.2 \\
0.1 & 0.3 & 0.2 & 1
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix} =
\begin{bmatrix}
0.2 \\
-2.6 \\
1.0 \\
-2.5
\end{bmatrix}
$$

In [104]:
# Question 16(a)
A16a = np.array([
    [10, -1, 2, 0],
    [-1, 11, -1, 3],
    [2, -1, 10, -1],
    [0, 3, -1, 8]
])
b16a = np.array([6, 25, -11, 15])

# Question 16(b)
A16b = np.array([
    [0, 5, -1, 2],
    [0, 8, -1, 1],
    [2, 1, -1, -1],
    [0, -1, -2, 1]
])
b16b = np.array([10, 16, 2, -2])

# Question 16(c)
A16c = np.array([
    [1, 0.5, -0.1, 0.1],
    [0.2, 1, -0.2, -0.1],
    [-0.1, -0.2, 1, 0.2],
    [0.1, 0.3, 0.2, 1]
])
b16c = np.array([0.2, -2.6, 1.0, -2.5])

# Systems
systems = [
    ("16(a)", A16a, b16a),
    ("16(b)", A16b, b16b),
    ("16(c)", A16c, b16c)
]

for name, A, b in systems:
    print(f"\nSolving {name}")
    
    # Gauss-Jacobi Method
    try:
        x_jacobi = gauss_jacobi_method(A, b, epsilon=10**-3)
        print(f"Gauss-Jacobi Method solution: {x_jacobi}")
        print(f"Residue: {residue(A,b,x_jacobi)}\n")
    except ValueError as e:
        print(f"Gauss-Jacobi failed: {e}")
    
    # Gauss-Seidel Method
    try:
        x_seidel = gauss_seidel_method(A, b, epsilon=10**-3)
        print(f"Gauss-Seidel Method solution: {x_seidel}")
        print(f"Residue: {residue(A,b,x_seidel)}\n")
    except ValueError as e:
        print(f"Gauss-Seidel failed: {e}")


Solving 16(a)
Gauss-Jacobi Method solution: [ 0.99967415  2.00044767 -1.00036916  1.00061919]
Residue: 0.007476969884709916

Gauss-Seidel Method solution: [ 0.99993981  1.99998904 -0.99997989  1.00000662]
Residue: 0.000550717702960668


Solving 16(b)
Gauss-Jacobi failed: Does not guarantee convergence
Gauss-Seidel failed: Does not guarantee convergence

Solving 16(c)
Gauss-Jacobi Method solution: [ 2.0001488  -2.9996819   1.00011304 -2.00003603]
Residue: 0.00032885499999979473

Gauss-Seidel Method solution: [ 1.99985925 -2.99997471  0.99999882 -1.99999328]
Residue: 0.00012732114370678227



#### 17. Consider the following linear system whose matrix of coefficients is sparse. (a)  the Gauss elimination method; (b) Apply the Gauss-Seidel method to the system with $ε = 10^{-3}$ or a limit of 10 iterations. (c) Comment on the results.

$$
\begin{bmatrix}
1 & 1 & -1 & 2 & -1 \\
2 & 0 & 0 & 0 & 0 \\
0 & 2 & 0 & 0 & 0 \\
4 & 0 & 0 & 16 & 0 \\
0 & 0 & 4 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4 \\
x_5
\end{bmatrix} =
\begin{bmatrix}
2 \\
2 \\
2 \\
20 \\
4
\end{bmatrix}
$$

In [108]:
# Define the system
A17 = np.array([
    [1, 1, -1, 2, -1],
    [2, 0, 0, 0, 0],
    [0, 2, 0, 0, 0], 
    [4, 0, 0, 16, 0],
    [0, 0, 4, 0, 0]
])
b17 = np.array([2, 2, 2, 20, 4])

print("Gauss Elimination method:")
try:
    x_gauss = gauss_elimination_method(A17, b17)
    print(f"Solution: {x_gauss}")
    print(f"Residue: {residue(A17, b17, x_gauss)}\n")
except ValueError as e:
    print(f"Failed: {e}\n")

print("Gauss-Seidel method:")
try:
    x_seidel = gauss_seidel_method(A17, b17, epsilon=1e-3)
    print(f"Solution: {x_seidel}")
    print(f"Residue: {residue(A17, b17, x_seidel)}")
except ValueError as e:
    print(f"Failed: {e}")

Gauss Elimination method:
Failed: Matrix is singular (no unique solution).

Gauss-Seidel method:
Failed: Does not guarantee convergence


#### 18. Show that the following system has no solution.

$$
\begin{bmatrix}
1 & 2 & 1 \\
2 & -1 & 2 \\
3 & 1 & 3
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix} =
\begin{bmatrix}
1 \\
2 \\
4
\end{bmatrix}
$$

In [ ]:
A18 = np.array([
    [1, 2, 1],
    [2, -1, 2],
    [3, 1, 3]
])
b18 = np.array([1, 2, 4])

try:
    x = gauss_elimination_method(A18, b18)
    print("Solution found (unexpected):", x)
except ValueError as e:
    print("As expected, no solution exists:", e)

#### 19. The following system of linear equations has a non-zero determinant:

$$
\begin{bmatrix}
0 & 15 & 1 & 3 \\
15 & 2 & 2 & 3 \\
0 & 4 & 15 & 1 \\
1 & 2 & 2 & 15
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
x_4
\end{bmatrix} =
\begin{bmatrix}
-3 \\
4 \\
7 \\
5
\end{bmatrix}
$$

#### (a) Find the solution using one of the direct numerical methods seen in class.

#### (b) Find the solution using one of the iterative numerical methods you saw in class, using as a stopping criterion $ε = 5 × 10^ {-2}$ or a limit of five iterations, and use $\frac{bi}{aii}$ as the initial solution. Remember to analyze the feasibility of applying the method.

In [115]:
A19 = np.array([
    [0, 15, 1, 3],
    [15, 2, 2, 3],
    [0, 4, 15, 1],
    [1, 2, 2, 15]
])
b19 = np.array([-3, 4, 7, 5])

# Part a) Direct method using LU decomposition
print("Direct method (LU decomposition):")
try:
    x_lu = lu_method(A19, b19)
    print(f"Solution: {x_lu}")
    print(f"Residue: {residue(A19, b19, x_lu)}\n")
except ValueError as e:
    print(f"Failed: {e}\n")

# Part b) Iterative method with ε = 5×10^-2
print("Iterative method (Gauss-Seidel):")
try:
    x_seidel = gauss_seidel_method(A19, b19, epsilon=5e-2)
    print(f"Solution: {x_seidel}")
    print(f"Residue: {residue(A19, b19, x_seidel)}")
except ValueError as e:
    print(f"Failed: {e}")

Direct method (LU decomposition):
Solution: [ 0.17758556 -0.2931345   0.5254681   0.29051648]
Residue: 8.881784197001252e-16

Iterative method (Gauss-Seidel):
Failed: Does not guarantee convergence


#### 20. Research and apply the Sassenfeld criterion in questions 17, 18 and 19.

In [116]:
def sassenfeld_criterion(A: np.ndarray) -> float:
    """
    Computes the Sassenfeld criterion for a given matrix A.
    Returns the maximum beta value. If max(beta) < 1, the method converges.
    """
    n = A.shape[0]
    beta = np.zeros(n)
    for i in range(n):
        s = 0
        for j in range(n):
            if j != i:
                s += abs(A[i, j]) * (beta[j] if j < i else 1)
        beta[i] = s / abs(A[i, i]) if A[i, i] != 0 else np.inf
    return np.max(beta)

# Apply Sassenfeld criterion to problems 17, 18, and 19
systems_20 = [
    ("17", A17),
    ("18", A18),
    ("19", A19)
]

for name, A in systems_20:
    beta_max = sassenfeld_criterion(A)
    print(f"Sassenfeld criterion for system {name}: max(beta) = {beta_max:.4f}")
    if beta_max < 1:
        print("=> The Gauss-Seidel method is guaranteed to converge.\n")
    else:
        print("=> The Gauss-Seidel method is NOT guaranteed to converge.\n")

Sassenfeld criterion for system 17: max(beta) = nan
=> The Gauss-Seidel method is NOT guaranteed to converge.

Sassenfeld criterion for system 18: max(beta) = 8.0000
=> The Gauss-Seidel method is NOT guaranteed to converge.

Sassenfeld criterion for system 19: max(beta) = nan
=> The Gauss-Seidel method is NOT guaranteed to converge.



C:\Users\Victor Xavier\AppData\Local\Temp\ipykernel_9468\1906654485.py:12: RuntimeWarning: invalid value encountered in scalar multiply
  s += abs(A[i, j]) * (beta[j] if j < i else 1)


### Give a computational solution to all the polynomial interpolation problems in the list of exercises, using the methods requested in each question.

#### 21. Consider the table below. Estimate $f(0.75)$ using linear interpolation and also using a polynomial of degree 2 (using Lagrange's method). Comment on the results.

$$
\begin{array}{|c|c|}
\hline
x & f(x) \\
\hline
0.5 & 1.2 \\
1.0 & 2.1 \\
1.5 & 3.3 \\
2.0 & 4.8 \\
\hline
\end{array}
$$

In [109]:
# Data points
x21 = np.array([0.5, 1.0, 1.5, 2.0])
y21 = np.array([1.2, 2.1, 3.3, 4.8])

# Linear interpolation using points [0.5, 1.0]
x_linear = x21[:2]  # Using first two points
y_linear = y21[:2]
linear_poly = lagrange_method(x_linear, y_linear)

# Quadratic interpolation using points [0.5, 1.0, 1.5]
x_quad = x21[:3]  # Using first three points  
y_quad = y21[:3]
quad_poly = lagrange_method(x_quad, y_quad)

x_eval = 0.75
print(f"Linear interpolation at x={x_eval}: {linear_poly(x_eval)}")
print(f"Quadratic interpolation at x={x_eval}: {quad_poly(x_eval)}")

Linear interpolation at x=0.75: 1.65
Quadratic interpolation at x=0.75: 1.6125000000000003


#### 22. A car is traveling in a straight line, the distances covered by which have been timed at various times. Given this data, use a polynomial interpolation of degree 3, in the form of a Newton form to determine the distance covered 15.6 minutes after the start.

$$
\begin{array}{|c|c|}
\hline
\text{Time (min)} & \text{Distance (km)} \\
\hline
0 & 0 \\
5 & 3.2 \\
10 & 6.8 \\
15 & 10.5 \\
20 & 14.7 \\
\hline
\end{array}
$$

In [110]:
time = np.array([0, 5, 10, 15, 20])
distance = np.array([0, 3.2, 6.8, 10.5, 14.7])

# Get Newton coefficients
coef = newton_div_diff(time, distance)

# Interpolate at t=15.6
t_eval = 15.6
distance_interp = newton_poly(coef, time, t_eval)
print(f"Interpolated distance at t={t_eval} min: {distance_interp:.2f} km")

Interpolated distance at t=15.6 min: 10.96 km


#### 23. It is suspected that the large amounts of tannin in mature oak leaves inhibit the growth of winter moth larvae (*Operophtera bromata L.,Geometridae*), which does a lot of damage to these trees in certain years. The following table lists the average weight of two samples of larvae at certain times in the first 28 days after hatching.

#### (a) Use some interpolatory method to determine a second degree polynomial and estimate what happens on day 7 and also at what point the average weight of the sample reaches 10g. Comment on the results.

#### (b) Can we use inverse interpolation in this tabulation?

$$
\begin{array}{|c|c|}
\hline
\text{Day} & \text{Weight (g)} \\
\hline
0 & 6.67 \\
6 & 17.33 \\
10 & 42.67 \\
13 & 37.33 \\
17 & 30.10 \\
\hline
\end{array}
$$

In [111]:
# Data points
days = np.array([0, 6, 10, 13, 17])
weights = np.array([6.67, 17.33, 42.67, 37.33, 30.10])

# Part a) Second degree polynomial using Lagrange
quad_poly = lagrange_method(days[:3], weights[:3])  # Using first 3 points

# Evaluate at day 7
day7 = quad_poly(7)
print(f"Estimated weight at day 7: {day7:.2f}g")

# Find when weight = 10g by evaluating at several points
t = np.linspace(0, 6, 1000)
weights_interp = [quad_poly(ti) for ti in t]
idx = np.argmin(np.abs(np.array(weights_interp) - 10))
time_10g = t[idx]
print(f"Weight reaches 10g at approximately day {time_10g:.2f}")

Estimated weight at day 7: 22.30g
Weight reaches 10g at approximately day 3.95


#### 24. Carbon dioxide is a fundamental gas for maintaining life on the planet. Without it, plants and other organisms would not carry out the process of photosynthesis, which transforms solar energy into chemical energy. This process is one of the phases of the so-called carbon cycle, which is vital for the maintenance of living beings. The excess carbon dioxide in the atmosphere is a direct result of burning fossil fuels, especially in the industrial and transportation sectors. According to the NOAA (National Oceanic and Atmospheric Administration), the concentration of CO2 in particles per million (ppm) in the atmosphere is as follows.

$$
\begin{array}{|c|c|}
\hline
\text{Year} & \text{CO₂ (ppm)} \\
\hline
1970 & 327 \\
1980 & 337 \\
1990 & 355 \\
2000 & 370 \\
2010 & 388 \\
\hline
\end{array}
$$

In [112]:
# Data for problem 24
years = np.array([1970, 1980, 1990, 2000, 2010])
co2 = np.array([327, 337, 355, 370, 388])

# Part a) Third degree interpolation for 2008
coef = newton_div_diff(years, co2)
co2_2008 = newton_poly(coef, years, 2008)
print(f"Estimated CO2 in 2008: {co2_2008:.1f} ppm")
print(f"Error from observed value (381 ppm): {abs(381-co2_2008):.1f} ppm")

# Part b) Find year when CO2 = 350 ppm using quadratic Lagrange
quad_poly = lagrange_method(years[:3], co2[:3])
# Search for root near 1988 (approximate visual guess)
t = np.linspace(1985, 1995, 1000)
co2_interp = [quad_poly(ti) for ti in t]
idx = np.argmin(np.abs(np.array(co2_interp) - 350))
year_350 = t[idx]
print(f"CO2 reached 350 ppm around year {year_350:.0f}")

Estimated CO2 in 2008: 383.3 ppm
Error from observed value (381 ppm): 2.3 ppm
CO2 reached 350 ppm around year 1988


#### (a) Determine by means of second-third interpolation what the estimated value is for the year 2008 and compare with the observed value, which was 381ppm.

#### (b) Using a second degree polynomial, determine in which year the concentration of CO2 in the atmosphere reached 350 ppm.

#### 25. The wind energy segment, which has been registering excellent expansion rates, should continue its performance in the coming years. The actual and estimated installed capacity of wind power in Brazil is shown in the table below (Source: Brazilian Wind Energy Association, 2014).

$$
\begin{array}{|c|c|}
\hline
\text{Year} & \text{Capacity (MW)} \\
\hline
2005 & 27 \\
2008 & 323 \\
2011 & 1430 \\
2014 & 7310 \\
\hline
\end{array}
$$

#### (a) Use second-degree interpolation to determine the installed capacity in 2010 and 2015 and comment on the reliability of these estimates.

#### (b) Determine, in a non-linear way, in which year the installed capacity was 5000 MW.

In [113]:
# Define the data points
years = np.array([2005, 2008, 2011, 2014])
capacity = np.array([27, 323, 1430, 7310])

# Part a) Second-degree interpolation for 2010 and 2015
quad_points = years[:3]  # Use first three points for quadratic interpolation
quad_capacity = capacity[:3]
quad_poly = lagrange_method(quad_points, quad_capacity)

# Evaluate at 2010 and 2015
capacity_2010 = quad_poly(2010)
capacity_2015 = quad_poly(2015)
print(f"Estimated capacity in 2010: {capacity_2010:.0f} MW")
print(f"Estimated capacity in 2015: {capacity_2015:.0f} MW")

# Part b) Find when capacity reached 5000 MW using Newton's method
# Use all points for better accuracy
coef = newton_div_diff(years, capacity)

# Search for 5000 MW crossing point using fine grid
t = np.linspace(2011, 2014, 1000)  # Search between 2011-2014 where 5000 likely occurred
capacities = [newton_poly(coef, years, ti) for ti in t]
idx = np.argmin(np.abs(np.array(capacities) - 5000))
year_5000 = t[idx]

print(f"\nCapacity reached 5000 MW around year {year_5000:.1f}")

# Comment on reliability
print("\nReliability Analysis:")
print("1. The growth is highly non-linear, making quadratic interpolation less reliable")
print("2. 2015 estimate may be less accurate as it's outside the interpolation range")
print("3. The rapid growth rate makes extrapolation particularly challenging")

Estimated capacity in 2010: 971 MW
Estimated capacity in 2015: 4168 MW

Capacity reached 5000 MW around year 2013.2

Reliability Analysis:
1. The growth is highly non-linear, making quadratic interpolation less reliable
2. 2015 estimate may be less accurate as it's outside the interpolation range
3. The rapid growth rate makes extrapolation particularly challenging
